# JAX Examples

## Basic JAX Benchmarking

ZeroBench automatically detects JAX arrays and optimizes benchmarking:

In [ ]:
import jax.numpy as jnp

from zerobench import Benchmark

bench = Benchmark()
x = jnp.ones(1000)
y = jnp.ones(1000)

with bench(method='add'):
    x + y

## JAX-Specific Report Fields

When JAX code is detected, the benchmark report includes additional fields:

In [ ]:
report = bench.to_dicts()[0]
print('Available fields:', list(report.keys()))

In [ ]:
print(f'First execution time: {report["first_execution_time"]:.2f} ns')
print(f'Compilation time: {report["compilation_time"]:.2f} ns')
print(f'Median execution time: {report["median_execution_time"]:.2f} ns')

## Visualizing the HLO

The `hlo` field contains the StableHLO representation of the compiled function. You can visualize it using `visu_hlo`:

In [ ]:
from visu_hlo import show

hlo_text = report['hlo']
show(hlo_text)

## Comparing Broadcasting Strategies

Benchmark different array operations to compare their performance:

In [ ]:
bench = Benchmark()

for N in [100, 1000, 10000]:
    x = jnp.ones(N)
    y = jnp.ones(1000)

    with bench(method='broadcast right', N=N):
        x[:, None] + y[None, :]

    with bench(method='broadcast left', N=N):
        x[None, :] + y[:, None]

In [ ]:
print(bench)

## Benchmarking JIT-compiled Functions

zerobench handles both regular and JIT-compiled JAX functions:

In [ ]:
import jax


@jax.jit
def matmul(a, b):
    return a @ b


bench = Benchmark()

for N in [64, 128, 256, 512]:
    a = jnp.ones((N, N))
    b = jnp.ones((N, N))

    with bench(operation='matmul', N=N):
        matmul(a, b)

In [ ]:
print(bench)

## Visualizing the Matrix Multiplication HLO

In [ ]:
# Get the HLO for the first benchmark (N=64)
report = bench.to_dicts()[0]
show(report['hlo'])

## Plotting Results

In [ ]:
bench.plot()